In [1]:
import numpy as np
import pandas as pd

import sys
sys.path.append('../')

import plotly.express as px
import plotly.graph_objects as go

# Reduce the dimensionality of the embedded series using PCA
from sklearn.decomposition import PCA


In [2]:
# list_data_name = ["BTCUSDT_1d", "ETHUSDT_1d", "BNBUSDT_1d", "XRPUSDT_1d", "ADAUSDT_1d", "LTCUSDT_1d"]                   
# list_data_name = ["BTCUSDT_1h", "ETHUSDT_1h", "BNBUSDT_1h", "XRPUSDT_1h", "ADAUSDT_1h", "LTCUSDT_1h"]                 
  
# list_data_name = ["ADAUSDT_1h"]                   
list_data_name = ["ADAUSDT_1d"]                   
                        
from_date = '2021-12-31 00:00:00' # 2020-12-31 00:00:00
until_date = '2025-07-31 00:00:00'
column_name = 'processed_log_return_wtmra_0'
list_delay_in_days = [1, 3, 7, 14, 21, 30]
dimension = 3  # Adjust as needed

In [3]:
# Lets implement time delay embedding

def time_delay_embedding(series, delay, dimension):
    embedded_data = np.array([series[i: i + delay * dimension: delay] for i in range(len(series) - delay * (dimension - 1))])
    return embedded_data

In [5]:
for data_name in list_data_name:
    
    # Importing data
    data = pd.read_csv(f'../data/01-output-{data_name}-from-{from_date}-until-{until_date}-log-return.csv', parse_dates=['date'], index_col='date')
    data = data[[column_name]]
    
    # keep original copy so each delay is computed on the same full series
    data_orig = data.copy()
    
    for delay in list_delay_in_days:
        # determine step (rows) from delay in days depending on data frequency in the name
        if data_name.endswith('_1h'):
            step = delay * 24
        else:
            # assume daily data (e.g. '_1d') -> one row per day
            step = delay

        # compute embedding on the original series
        embedded_series = time_delay_embedding(data_orig[column_name].values, step, dimension)
        if embedded_series.size == 0:
            print(f"Skipping delay={delay} (step={step}): not enough data points for embedding")
            continue

        # Apply PCA to reduce dimensionality
        pca = PCA(n_components=1)
        pca.fit(embedded_series)
        reduced_series = pca.transform(embedded_series)

        # add in the beginning of reduced_series the required NaN values so length matches original series
        pad_count = (dimension - 1) * step
        reduced_series_new = np.concatenate((np.full((pad_count, 1), np.nan), reduced_series), axis=0)
        data_orig[f'time_delay_{delay}_in_days_dimension_{dimension}_dimension_embedding_pca_1'] = reduced_series_new    

    # remove rows with any NaN (after all delays computed) and export
    data_to_export = data_orig.dropna()
    data_to_export.to_csv(f'../data/02e-output-{data_name}-log-return-time-delay-embedding.csv')